CSCE 4143: Data Mining\
Xintao Wu, Ph.D.\
Practice Project\
Group 7: Levi Crider, Caleb Holmes, Spencer Smith, and Ethan Weems\
Step 2: Clustering

Import Packages

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.cluster import KMeans
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

Import Data from Adult Data File

In [ ]:
labels = ["age", "workclass", "fnlwgt", "education", "education-num", "marital-status", "occupation", "relationship", "race", "sex", "capital-gain", "capital-loss", "hours-per-week", "native-country", "actual-value"]
numeric_labels = ["age", "fnlwgt", "education-num", "capital-gain", "capital-loss", "hours-per-week"]
df = pd.read_table("./adult/adult.data", names=labels, delimiter=",")

Clean Data\
-Remove unknown values\
-Split numerical attributes into binary attributes

In [ ]:
df = df.astype(str)

# using list comprehension
curr_labels = [i for i in labels if i not in numeric_labels]

substring = '\?' # character indicating null value
for label in curr_labels:
   filter = df[label].str.contains(substring) 
   df = df[~filter]

df = df.reset_index(drop=True)

df[numeric_labels] = df[numeric_labels].astype(float)

mean_values = df[numeric_labels].mean()

for num_label, threshold in mean_values.items():
    df[num_label] = df[num_label].apply(lambda x: 1 if x >= threshold else 0)

df[numeric_labels]

Perform One-Hot Encoding and Split Data

In [ ]:
X = df[curr_labels[0 : len(curr_labels) - 1]]
y = df["actual-value"].values

one_hot = OneHotEncoder()
label_encoder = LabelEncoder()
x_encode = one_hot.fit_transform(X).toarray()
y_encode = label_encoder.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(x_encode, y_encode, test_size=0.2)


Build K-means Clustering Algorithm 

In [ ]:
def createKMeansCluster(num):
    # create and train model
    kmeans = KMeans(n_clusters=num, random_state=42)
    kmeans.fit(X_train)
    
    centroids = kmeans.cluster_centers_
    
    return centroids

3 Clusters

In [ ]:
createKMeansCluster(3)

5 Clusters

In [ ]:
createKMeansCluster(5)

10 Clusters

In [ ]:
createKMeansCluster(10)

K-Nearest Neighbor Algorithm

In [ ]:
def kNN_Wrapper(num):
    # use last 10 records from test data
    data = X_test[len(X_test) - 10: len(X_test)]
    
    knn = KNeighborsClassifier(n_neighbors=num)
    
    knn.fit(X_train, y_train)
    
    predictions = knn.predict(data)
    predictions
    
    accuracy = accuracy_score(y_test[len(y_test) - 10 : len(y_test)], predictions)
    print(f"Accuracy: {accuracy:.2f}")
    
    print(classification_report(y_test[len(y_test) - 10 : len(y_test)], predictions))

kNN 3

In [ ]:
kNN_Wrapper(3)

kNN 5

In [ ]:
kNN_Wrapper(5)

kNN 10

In [ ]:
kNN_Wrapper(10)

<h1>Task #3:</h3>
Build SVM classifier and report the predicted accuracy of the test data.

In [ ]:
from sklearn import svm
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score

In [ ]:
# instantiate the model using sklearn library
svm_model = svm.SVC()
# train the model on the training data defined previously
svm_model.fit(X_train, y_train)

In [ ]:
# evaluate performance of model using unseen test data
svm_predictions = svm_model.predict(X_test)

Compare predictions to the true values

In [ ]:
def plot_confusion_matrix(predictions, true):
    confMat = confusion_matrix(true, predictions)
    dispConfMat = ConfusionMatrixDisplay(confusion_matrix=confMat)
    dispConfMat.plot()
    plt.title("Confusion Matrix")
    plt.show()
    accuracy = accuracy_score(y_true=true, y_pred=predictions)
    print("The accuracy of the model is", accuracy)


In [ ]:
plot_confusion_matrix(predictions=svm_predictions,true=y_test)

<h1>Task #4:</h3>
Build a neural network classifier and report the predicted accuracy of the test data.

In [ ]:
from sklearn.neural_network import MLPClassifier

In [ ]:
# instantiate the model using sklearn library
nn_model = MLPClassifier(random_state=77, max_iter=500)
# train the model using the training data defined previously
nn_model.fit(X_train, y_train)

In [ ]:
# evaluate the model using the previously unseen testing data
nn_predictions = nn_model.predict(X_test)

In [ ]:
plot_confusion_matrix(predictions=nn_predictions,true=y_test)